In [1]:
# this script adds Sales Month and Sales Quarter in unified format to all data until and including 2024 Q4

import pandas as pd
import os
import re

# test commit und dann noch mehr

input1 = "../../50 KM Group/Royalties/Statements/Karen/Earth/Combined statements/old/Earth_2022Q4_2024Q4_1_raw_combined_fixed.csv"
outputfilename = "../../50 KM Group/Royalties/Statements/Karen/_output/Earth_2022Q4_2024Q4_1_raw_combined_Salesmonthandquarter_elena.csv"

# tried something else

In [2]:
df = pd.read_csv(input1)

print(f"DataFrame: Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Total fee: {df['Royalties (USD)'].sum()}. Total units: {df['Units'].sum()}")

df = df.drop(columns=['实际分成收入(TWD)','总计','Payable CNY'],errors = 'ignore')
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df = df.rename(columns={"Statement": "Statement Quarter"})

def clean_period(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return str(int(value))
    s = str(value).strip().replace("\n", "").replace("\r", "")
    s = re.sub(r"\.0$", "", s)
    return s
    # return str(value).strip().replace("\n", "").replace("\r", "").replace(r"\.0$", "", regex=True)
    
df["period_cleaned"] = df["Period"].apply(clean_period)
df["period_start_cleaned"] = df["Period start"].apply(clean_period)


/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_25263/2658576496.py:1: DtypeWarning: Columns (0,2,6,7,12,13,16,17,18,21,22,27,28,30,33,34,37,39,40,41,42,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input1)


DataFrame: Rows: 991290, Columns: 56
Total fee: 378369.37166618125. Total units: 239471829.0


In [3]:
def choose_period(row):
    period = row["period_cleaned"]
    period_start = row["period_start_cleaned"]

    # Helper function to test if value is 'filled'
    def is_filled(x):
        if pd.isna(x):
            return False
        if isinstance(x, str) and x.strip() == "":
            return False
        return True

    has_period = is_filled(period)
    has_period_start = is_filled(period_start)

    # Both empty
    if not has_period and not has_period_start:
        return None

    # Both filled
    if has_period and has_period_start:
        raise ValueError(f"Both Period and Period start are filled in row {row.name}: Period={period}, Period start={period_start}")

    # Only one filled
    if has_period:
        return period
    else:
        return period_start

# Apply the function
df["period_final"] = df.apply(choose_period, axis=1)

In [4]:
print(df["period_final"].map(type).value_counts())

period_final
<class 'str'>         974830
<class 'NoneType'>     16460
Name: count, dtype: int64


In [5]:
# Define a function to parse various formats
def parse_date(value):
    value = str(value).strip()
    #fmts = ["%Y/%m/%d", "%Y%m%d", "%Y%m", "%Y %m"]
    fmts = ["%Y/%m/%d", "%Y%m%d", "%m/%d/%Y", "%Y%m", "%Y %m", "%Y-%m-%d %H:%M:%S"]

    for fmt in fmts:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    # If parsing fails, return NaT
    return pd.NaT

# Apply parsing to each column
df["period_parsed"] = df["period_final"].apply(parse_date)

In [6]:
unique_cleaned = df["period_parsed"].astype(str).apply(repr).unique()
for u in unique_cleaned:
    print(repr(u))

"'2023-10-01'"
"'2023-01-01'"
"'2023-01-02'"
"'2023-09-01'"
"'2023-08-01'"
"'2023-07-01'"
"'2023-06-01'"
"'2023-03-01'"
"'2023-05-01'"
"'2023-04-01'"
"'NaT'"
"'2023-02-01'"
"'2022-01-02'"
"'2022-01-01'"
"'2024-01-01'"
"'2024-02-01'"
"'2024-03-01'"
"'2023-11-01'"
"'2023-12-01'"
"'2024-04-01'"
"'2024-05-01'"
"'2024-06-01'"
"'2024-07-01'"
"'2024-08-01'"
"'2024-09-01'"
"'2019-07-07'"
"'2024-10-01'"
"'2024-01-02'"
"'2024-11-01'"
"'2024-12-01'"


In [7]:
df['period_parsed'].map(type).value_counts()

period_parsed
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    974444
<class 'pandas._libs.tslibs.nattype.NaTType'>          16846
Name: count, dtype: int64

In [8]:
# You can pick which column to use (e.g., Period start first if available), here as an example:
# df["Sales_date"] = df["Period_start_dt"].combine_first(df["Period_dt"])

# Format Sales Month: "YYYY MM"

df["Sales_date"] = pd.to_datetime(df["period_parsed"], format="%Y%m", errors="coerce")

df["Sales Month"] = df["Sales_date"].dt.strftime("%Y %m")

# Format Sales Quarter: "YYYY QN"
df["Sales Quarter"] = (
    df["Sales_date"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
)

# Special override for "2453年11月"
mask_special = df["Period"] == "2453年11月"

df.loc[mask_special, "Sales Quarter"] = "2023 Q3"
df.loc[mask_special, "Sales Month"] = "9999"

df["Sales Quarter"] = df["Sales Quarter"].where(
    df["Sales Quarter"] != "NaT",
    df["Statement Quarter"]
)

df['Sales Quarter']=df['Sales Quarter'].str.replace('NaT','9999')
df.fillna({'Sales Month': '9999'}, inplace=True)

df.rename(columns={"Statement": "Statement Quarter"}, inplace=True)
df = df.drop(columns=['Period_dt','Period_start_dt','Sales_date'],errors = 'ignore')
df = df.sort_index(axis=1)
print(f"Total fee: {df['Royalties (USD)'].sum()}. Total units: {df['Units'].sum()}")

Total fee: 378369.37166618125. Total units: 239471829.0


In [9]:
#df.to_csv(outputfilename, index=False)

In [10]:
#df.head(1000).to_csv("test_output", index=False)
df[["Period", "Period start", "Sales Quarter", "Sales Month"]].to_csv(outputfilename, index=False)
print(f"DataFrame: Rows: {df.shape[0]}, Columns: {df.shape[1]}")


DataFrame: Rows: 991290, Columns: 51
